# Practical Task 6 - Interactive Data Visualization Dashboard

## 1. Introduction

**Interactive data visualization** is a method of graphical representation that allows users to engage with the data. Unlike static charts, interactive visualizations enable users to zoom, filter, and hover over data points to uncover deeper insights. This approach is essential for modern data analysis as it transforms complex datasets into intuitive, explorable interfaces.

The **purpose of this dashboard** is to explore the **Global Country Information (2023)** dataset. By leveraging interactive elements, we aim to identify global patterns in population distribution, economic indicators, and land usage across different nations. This dashboard will provide a comprehensive view of global metrics through various chart types, including geographic maps and 3D plots.

## 2. Import Libraries



In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

## 3. Load Data


In [2]:
# Load the dataset
df = pd.read_csv("data.csv")

# Display the first few rows
df.head()

,Country,Density\n(P/Km2),Abbreviation,Agricultural Land( %),Land Area(Km2),Armed Forces size,Birth Rate,Calling Code,Capital/Major City,Co2-Emissions,...,Out of pocket health expenditure,Physicians per thousand,Population,Population: Labor force participation (%),Tax revenue (%),Total tax rate,Unemployment rate,Urban_population,Latitude,Longitude
0,Afghanistan,60,AF,58.10%,"652,230","323,000",32.49,93.0,Kabul,"8,672",...,78.40%,0.28,"38,041,754",48.90%,9.30%,71.40%,11.12%,"9,797,273",33.939110,67.709953
1,Albania,105,AL,43.10%,"28,748","9,000",11.78,355.0,Tirana,"4,536",...,56.90%,1.20,"2,854,191",55.70%,18.60%,36.60%,12.33%,"1,747,593",41.153332,20.168331
2,Algeria,18,DZ,17.40%,"2,381,741","317,000",24.28,213.0,Algiers,"150,006",...,28.10%,1.72,"43,053,054",41.20%,37.20%,66.10%,11.70%,"31,510,100",28.033886,1.659626
3,Andorra,164,AD,40.00%,468,NaN,7.20,376.0,Andorra la Vella,469,...,36.40%,3.33,"77,142",NaN,NaN,NaN,NaN,"67,873",42.506285,1.521801
4,Angola,26,AO,47.50%,"1,246,700","117,000",40.73,244.0,Luanda,"34,693",...,33.40%,0.21,"31,825,295",77.50%,9.20%,49.10%,6.89%,"21,061,025",-11.202692,17.873887


## 4. Data Cleaning & Preparation


In [3]:
# Define a helper function to clean numeric columns
def clean_numeric(val):
    if isinstance(val, str):
        # Remove commas, percentage signs, and dollar signs
        val = val.replace(",", "").replace("%", "").replace("$", "").strip()
    try:
        return float(val)
    except:
        return np.nan

# List of columns to clean
cols_to_clean = [
    "Agricultural Land( %)", "Land Area(Km2)", "Armed Forces size", 
    "Co2-Emissions", "CPI Change (%)", "Gasoline Price", "GDP", 
    "Minimum wage", "Out of pocket health expenditure", 
    "Population", "Population: Labor force participation (%)", 
    "Tax revenue (%)", "Total tax rate", "Unemployment rate", 
    "Urban_population", "Latitude", "Longitude"
]

# Apply cleaning
for col in cols_to_clean:
    if col in df.columns:
        df[col] = df[col].apply(clean_numeric)

# Handle missing values by dropping rows where critical mapping data is missing
df = df.dropna(subset=["Country", "Latitude", "Longitude"])

# Fill other missing numeric values with 0 or median for visualization purposes
df = df.fillna(0)

print("Data cleaning complete. Sample of cleaned data:")
df[["Country", "GDP", "Urban_population", "Total tax rate"]].head()

Data cleaning complete. Sample of cleaned data:


,Country,GDP,Urban_population,Total tax rate
0,Afghanistan,1.910135e+10,9797273.0,71.4
1,Albania,1.527808e+10,1747593.0,36.6
2,Algeria,1.699882e+11,31510100.0,66.1
3,Andorra,3.154058e+09,67873.0,0.0
4,Angola,9.463542e+10,21061025.0,49.1


## 5. Interactive Line Chart



In [4]:
df_sorted = df.sort_values(by="Birth Rate")
fig_line = px.line(df_sorted, x="Country", y=["Birth Rate", "Life expectancy"], 
                  title="Birth Rate and Life Expectancy by Country (Sorted by Birth Rate)",
                  labels={"value": "Rate / Age", "variable": "Metric"},
                  hover_name="Country")
fig_line.update_layout(xaxis_tickangle=-45)
fig_line.show()

## 6. Interactive Bar Chart



In [5]:
top_10_urban = df.nlargest(10, "Urban_population")
fig_bar = px.bar(top_10_urban, x="Country", y="Urban_population", 
                 color="Urban_population", 
                 title="Top 10 Countries by Urban Population",
                 labels={"Urban_population": "Urban Population"},
                 color_continuous_scale="Viridis")
fig_bar.show()

## 7. Interactive Scatter Plot



In [6]:
fig_scatter = px.scatter(df, x="GDP", y="Unemployment rate", 
                         size="Population", color="Country", 
                         hover_name="Country", log_x=True, 
                         title="GDP vs. Unemployment Rate (Size = Population)",
                         labels={"GDP": "GDP (Log Scale)", "Unemployment rate": "Unemployment Rate (%)"})
fig_scatter.show()

## 8. Choropleth Map (Geographic Visualization)


In [7]:
fig_map = px.choropleth(df, locations="Country", locationmode="country names", 
                        color="Total tax rate", 
                        hover_name="Country", 
                        title="Global Distribution of Total Tax Rate",
                        color_continuous_scale="Reds")
fig_map.show()

/tmp/ipykernel_671087/77731232.py:1: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig_map = px.choropleth(df, locations="Country", locationmode="country names",


## 9. 3D Scatter Plot


In [8]:
fig_3d = px.scatter_3d(df, x="Population", y="Total tax rate", z="Unemployment rate", 
                       color="Country", size_max=18, opacity=0.7, 
                       title="3D Analysis: Population, Tax Rate, and Unemployment",
                       labels={"Population": "Population", "Total tax rate": "Tax Rate", "Unemployment rate": "Unemployment"})
fig_3d.show()

## 10. Time Series Animation (Simulation)



In [9]:
# Simulate 5 years of data based on current urban population with slight growth
years = [2023, 2024, 2025, 2026, 2027]
animated_data = []

for year in years:
    temp_df = df[["Country", "Urban_population"]].copy()
    growth_factor = 1 + (year - 2023) * 0.02 # 2% annual growth simulation
    temp_df["Urban_population"] *= growth_factor
    temp_df["Year"] = year
    animated_data.append(temp_df)

df_animated = pd.concat(animated_data)

fig_anim = px.bar(df_animated.nlargest(50, "Urban_population"), 
                  x="Country", y="Urban_population", 
                  animation_frame="Year", 
                  range_y=[0, df_animated["Urban_population"].max() * 1.1],
                  title="Simulated Urban Population Growth (2023-2027)")
fig_anim.show()

## 11. Dashboard Layout



In [10]:
# Create subplots for all visualizations
fig_dash = make_subplots(
    rows=3, cols=2, 
    subplot_titles=("Birth Rate and Life Expectancy", "Top 10 Urban Populations", "GDP vs Unemployment", "Global Tax Rate", "3D Population, Tax, Unemployment", "Simulated Urban Population Growth"),
    specs=[[{"type": "xy"}, {"type": "bar"}],
           [{"type": "scatter"}, {"type": "choropleth"}],
           [{"type": "scene"}, {"type": "bar"}]]
)

# Add Line Chart
for trace in fig_line.data:
    fig_dash.add_trace(trace, row=1, col=1)

# Add Bar Chart
for trace in fig_bar.data:
    fig_dash.add_trace(trace, row=1, col=2)

# Add Scatter Plot
for trace in fig_scatter.data:
    fig_dash.add_trace(trace, row=2, col=1)

# Add Choropleth Map
for trace in fig_map.data:
    fig_dash.add_trace(trace, row=2, col=2)

# Add 3D Scatter Plot
for trace in fig_3d.data:
    fig_dash.add_trace(trace, row=3, col=1)

# Add Animated Bar Chart
for trace in fig_anim.data:
    fig_dash.add_trace(trace, row=3, col=2)

fig_dash.update_layout(height=1200, showlegend=False, title_text="Global Country Insights Dashboard 2023")
fig_dash.show()

## 12. Export Dashboard



In [11]:
# Save the combined dashboard
fig_dash.write_html("dashboard.html")
print("Dashboard exported successfully as dashboard.html")

Dashboard exported successfully as dashboard.html


## 13. Conclusion

This interactive dashboard provides a multi-faceted view of global country statistics for 2023. Key insights include:
- **Urbanization:** A few countries dominate the global urban population landscape.
- **Economy:** There is a visible relationship between GDP and unemployment, though outliers exist.
- **Taxation:** Geographic mapping reveals significant regional differences in tax policies.
- **Interactivity:** The use of Plotly allows for dynamic exploration, making the data more accessible than static representations.